In [1]:
# ЯЧЕЙКА 1 — Гиперпараметры и конфигурация


import os
import torch

# --- Пути ---
LF_PATH        = "lf.npy"          # реальный двухканальный снимок для инференса
CHECKPOINT_DIR = "checkpoints13"     # папка для сохранения весов
VIZ_DIR        = "viz"             # папка для визуализаций в процессе обучения

# --- Датасет ---
TRAIN_RATIO    = 0.85              # доля обучающей выборки (если не задана выше)
IMG_SIZE       = 256               # размер кропа при обучении (кратно 32)
AUGMENT        = True              # включить аугментации (горизонтальный флип)

# --- Синтетические полосы ---
STRIPE_MIN_COUNT = 3               # минимальное количество полос на изображение
STRIPE_MAX_COUNT = 18              # максимальное количество полос
STRIPE_MIN_WIDTH = 1               # минимальная ширина полосы (пикс.)
STRIPE_MAX_WIDTH = 10              # максимальная ширина полосы (пикс.)
STRIPE_MAX_ANGLE = 5               # максимальный наклон полосы (градусы)
STRIPE_INTENSITY_RANGE = (0.1, 0.5)  # диапазон интенсивности полос
STRIPE_BREAK_PROB = 0.3            # вероятность, что полоса прерывается

# --- Обучение ---
BATCH_SIZE     = 16
NUM_WORKERS    = 4
NUM_EPOCHS     = 60
LR             = 2e-4
WEIGHT_DECAY   = 1e-5
SCHEDULER_PATIENCE = 5            # ReduceLROnPlateau: сколько эпох ждать
VIZ_EVERY      = 5                # визуализировать каждые N эпох
N_VIZ_SAMPLES  = 4               # сколько примеров выводить

# --- Функция потерь ---
LAMBDA_L1      = 1.0
LAMBDA_SSIM    = 0.3
LAMBDA_PERC    = 0.05

# --- Устройство ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {DEVICE} | GPU count: {torch.cuda.device_count()}")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(VIZ_DIR, exist_ok=True)

Устройство: cuda | GPU count: 2


In [2]:

import numpy as np
import math
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision import models

from PIL import Image
import matplotlib.pyplot as plt
from skimage.draw import polygon          # для наклонных полос
from scipy.ndimage import rotate as nd_rotate

In [3]:
# ЯЧЕЙКА 3 — Генератор синтетических полос


def add_stripes(img_np: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    Добавляет синтетические горизонтальные полосы (возможно наклонные)
    к двухканальному изображению [2, H, W] в диапазоне [0, 1].

    Полосы добавляются аддитивно (с clamp в [0,1]).
    Имитирует артефакты сканирования породы.
    """
    C, H, W = img_np.shape
    result   = img_np.copy()

    n_stripes = rng.integers(STRIPE_MIN_COUNT, STRIPE_MAX_COUNT + 1)

    for _ in range(n_stripes):
        width     = rng.integers(STRIPE_MIN_WIDTH, STRIPE_MAX_WIDTH + 1)
        intensity = rng.uniform(*STRIPE_INTENSITY_RANGE)
        angle_deg = rng.uniform(-STRIPE_MAX_ANGLE, STRIPE_MAX_ANGLE)
        y_center  = rng.integers(0, H)

        # Строим маску полосы на большом холсте, потом поворачиваем
        pad = int(W * math.tan(math.radians(abs(angle_deg))) + width + 1)
        canvas = np.zeros((H + 2 * pad, W + 2 * pad), dtype=np.float32)

        y0 = y_center + pad - width // 2
        y1 = y0 + width
        y0 = max(0, min(y0, canvas.shape[0]))
        y1 = max(0, min(y1, canvas.shape[0]))
        canvas[y0:y1, :] = 1.0

        # Поворот маски
        rotated = nd_rotate(canvas, angle_deg, reshape=False, order=1)
        mask    = rotated[pad:pad + H, pad:pad + W]

        # Случайные разрывы
        if rng.random() < STRIPE_BREAK_PROB:
            gap_start = rng.integers(0, W - 1)
            gap_len   = rng.integers(W // 10, W // 3)
            mask[:, gap_start:gap_start + gap_len] = 0.0

        # Случайная интенсивность по каналам (R и G могут отличаться)
        for c in range(C):
            c_intensity = intensity * rng.uniform(0.7, 1.3)
            result[c]   = np.clip(result[c] + mask * c_intensity, 0.0, 1.0)

    return result

In [4]:
import numpy as np
from olimp.dataset.olimp import olimp
from olimp.dataset import read_img_path

CATEGORIES = [
    'abstracts and textures', 'abstracts and textures/abstract art',
    'abstracts and textures/backgrounds and patterns',
    'abstracts and textures/colorful abstracts',
    'abstracts and textures/geometric shapes',
    'abstracts and textures/neon abstracts', 'abstracts and textures/textures',
    'animals', 'animals/birds', 'animals/farm animals',
    'animals/insects and spiders', 'animals/marine life', 'animals/pets',
    'animals/wild animals', 'art and culture',
    'art and culture/cartoon and comics',
    'art and culture/crafts and handicrafts',
    'art and culture/dance and theater performances',
    'art and culture/music concerts and instruments',
    'art and culture/painting and frescoes',
    'art and culture/sculpture and bas-reliefs', 'food and drinks',
    'food and drinks/desserts and bakery', 'food and drinks/dishes',
    'food and drinks/drinks',
    'food and drinks/food products on store shelves',
    'food and drinks/fruits and vegetables', 'food and drinks/street food',
    'interiors', 'interiors/gyms and pools', 'interiors/living spaces',
    'interiors/museums and galleries', 'interiors/offices',
    'interiors/restaurants and cafes',
    'interiors/shopping centers and stores', 'nature', 'nature/beaches',
    'nature/deserts', 'nature/fields and meadows', 'nature/forest',
    'nature/mountains', 'nature/water bodies', 'objects and items',
    'objects and items/books and stationery',
    'objects and items/clothing and accessories',
    'objects and items/electronics and gadgets',
    'objects and items/furniture and decor',
    'objects and items/tools and equipment',
    'objects and items/toys and games', 'portraits and people',
    'portraits and people/athletes and dancers',
    'portraits and people/crowds and demonstrations',
    'portraits and people/group photos',
    'portraits and people/individual portraits',
    'portraits and people/models on runway',
    'portraits and people/workers in their workplaces',
    'sports and active leisure',
    'sports and active leisure/cycling and rollerblading',
    'sports and active leisure/extreme sports',
    'sports and active leisure/individual sports',
    'sports and active leisure/martial arts',
    'sports and active leisure/team sports',
    'sports and active leisure/tourism and hikes', 'text and pictogram',
    'text and pictogram/billboard text', 'text and pictogram/blueprints',
    'text and pictogram/caricatures and pencil drawing',
    'text and pictogram/text documents', 'text and pictogram/traffic signs',
    'urban scenes', 'urban scenes/architecture',
    'urban scenes/city at night', 'urban scenes/graffiti and street art',
    'urban scenes/parks and squares', 'urban scenes/streets and avenues',
    'urban scenes/transport',
]


all_paths = []
for cat in CATEGORIES:
    try:
        ds    = olimp(categories={cat})
        paths = ds[cat]
        all_paths.extend(paths)
    except Exception as e:
        print(f"  {cat}: пропущено ({e})")

rng = np.random.default_rng(42)
rng.shuffle(all_paths)
all_paths = all_paths[:1000]
print(f"\nИтого загружено: {len(all_paths)} изображений")

n_train     = int(TRAIN_RATIO * len(all_paths))
train_paths = all_paths[:n_train]
val_paths   = all_paths[n_train:]
print(f"Train: {len(train_paths)} изображений | Val: {len(val_paths)} изображений")

/home/user/.local/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')


Итого загружено: 1000 изображений
Train: 850 изображений | Val: 150 изображений


In [5]:
# ЯЧЕЙКА 4 —  Dataset


class StripeRemovalDataset(Dataset):
    """
    Dataset для обучения модели удаления полос.

    Загружает трёхканальные RGB-изображения из OLIMP,
    оставляет только каналы R и G,
    нормализует в [0, 1],
    добавляет синтетические полосы.

    Возвращает пару: (испорченное_изображение, чистое_изображение)
    Оба тензора: float32, shape [2, IMG_SIZE, IMG_SIZE], диапазон [0, 1].
    """

    def __init__(self, paths, augment: bool = False, seed: int = 0):
        self.paths   = paths
        self.augment = augment
        # Каждый worker получит свой rng через worker_init_fn
        self.base_seed = seed

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]

        # --- Загрузка изображения ---
        try:
            img = Image.open(str(path)).convert("RGB")
        except Exception:
            # Если файл повреждён — возвращаем пустую пару
            dummy = torch.zeros(2, IMG_SIZE, IMG_SIZE)
            return dummy, dummy

        # --- Случайный кроп до IMG_SIZE x IMG_SIZE ---
        w, h = img.size
        if w < IMG_SIZE or h < IMG_SIZE:
            img = TF.resize(img, (max(h, IMG_SIZE), max(w, IMG_SIZE)))
            w, h = img.size

        # Случайная обрезка
        rng_torch = torch.Generator()
        rng_torch.manual_seed(self.base_seed + idx)
        i = torch.randint(0, h - IMG_SIZE + 1, (1,), generator=rng_torch).item()
        j = torch.randint(0, w - IMG_SIZE + 1, (1,), generator=rng_torch).item()
        img = TF.crop(img, i, j, IMG_SIZE, IMG_SIZE)

        # --- Горизонтальный флип (аугментация) ---
        if self.augment and torch.rand(1, generator=rng_torch).item() > 0.5:
            img = TF.hflip(img)

        # --- Конвертация: оставляем только R и G ---
        img_np = np.array(img, dtype=np.float32) / 255.0   # [H, W, 3]
        clean  = np.stack([img_np[:, :, 0],                 # R
                           img_np[:, :, 1]], axis=0)        # G  → [2, H, W]

        # --- Синтетические полосы ---
        rng_np  = np.random.default_rng(self.base_seed + idx * 31337)
        noisy   = add_stripes(clean, rng_np)

        clean_t = torch.from_numpy(clean)
        noisy_t = torch.from_numpy(noisy)

        return noisy_t, clean_t


def worker_init_fn(worker_id):
    """Разные seed для каждого DataLoader-worker'а."""
    np.random.seed(torch.initial_seed() % (2**32) + worker_id)
    random.seed(torch.initial_seed() % (2**32) + worker_id)


# --- Создание датасетов и загрузчиков ---
train_dataset = StripeRemovalDataset(train_paths, augment=AUGMENT, seed=42)
val_dataset   = StripeRemovalDataset(val_paths,   augment=False,   seed=0)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# --- Быстрая проверка ---
noisy_ex, clean_ex = next(iter(train_loader))
print(f"Пример: noisy {noisy_ex.shape}, clean {clean_ex.shape}, "
      f"min={noisy_ex.min():.3f}, max={noisy_ex.max():.3f}")

Train batches: 53 | Val batches: 10
Пример: noisy torch.Size([16, 2, 256, 256]), clean torch.Size([16, 2, 256, 256]), min=0.000, max=1.000


In [6]:
# ЯЧЕЙКА 5 — Строительные блоки нейросети


# ---- Squeeze-and-Excitation блок (канальное внимание) 
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


# ---- Residual блок с SE-вниманием ----
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.InstanceNorm2d(channels, affine=True),
        )
        self.se  = SEBlock(channels)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(x + self.se(self.block(x)))


# ---- Encoder-блок (Down) ----
class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_res=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.res  = nn.Sequential(*[ResidualBlock(out_ch) for _ in range(n_res)])
        self.down = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1, bias=False)

    def forward(self, x):
        x = self.res(self.conv(x))
        return x, self.down(x)   # skip, downsampled


# ---- Attention Gate (для decoder) ----
class AttentionGate(nn.Module):
    def __init__(self, g_ch, x_ch, int_ch):
        super().__init__()
        self.Wg = nn.Conv2d(g_ch, int_ch, 1, bias=False)
        self.Wx = nn.Conv2d(x_ch, int_ch, 1, bias=False)
        self.psi = nn.Sequential(
            nn.Conv2d(int_ch, 1, 1, bias=False),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        # g — из decoder (gate signal), x — skip connection
        psi = self.relu(self.Wg(g) + self.Wx(x))
        psi = self.psi(psi)
        return x * psi


# ---- Decoder-блок (Up) ----
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_res=2):
        super().__init__()
        self.up  = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.ag  = AttentionGate(out_ch, skip_ch, out_ch // 2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.res = nn.Sequential(*[ResidualBlock(out_ch) for _ in range(n_res)])

    def forward(self, x, skip):
        x    = self.up(x)
        skip = self.ag(x, skip)
        x    = torch.cat([x, skip], dim=1)
        return self.res(self.conv(x))

In [7]:
# ЯЧЕЙКА 6 — Архитектура U-Net с Residual + Attention


class ResAttUNet(nn.Module):
    """
    U-Net с:
    - Residual блоками в каждом уровне
    - SE (Squeeze-and-Excitation) вниманием внутри Residual
    - Attention Gates в skip-соединениях
    - Instance Normalization (хорошо работает на малых батчах)
    - Reflection padding в bottleneck (меньше граничных артефактов)

    Вход: [B, 2, H, W] — 2 канала (R, G)
    Выход: [B, 2, H, W] — восстановленное изображение
    """

    def __init__(self, in_ch=2, out_ch=2, base_ch=64):
        super().__init__()

        # Encoder
        self.enc1 = EncoderBlock(in_ch,       base_ch,     n_res=2)
        self.enc2 = EncoderBlock(base_ch,     base_ch*2,   n_res=2)
        self.enc3 = EncoderBlock(base_ch*2,   base_ch*4,   n_res=3)
        self.enc4 = EncoderBlock(base_ch*4,   base_ch*8,   n_res=3)

        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(base_ch*8, base_ch*16, 3, bias=False),
            nn.InstanceNorm2d(base_ch*16, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            ResidualBlock(base_ch*16),
            ResidualBlock(base_ch*16),
            ResidualBlock(base_ch*16),
            nn.ReflectionPad2d(1),
            nn.Conv2d(base_ch*16, base_ch*8, 3, bias=False),
            nn.InstanceNorm2d(base_ch*8, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # Decoder
        self.dec4 = DecoderBlock(base_ch*8,  base_ch*8, base_ch*4, n_res=2)
        self.dec3 = DecoderBlock(base_ch*4,  base_ch*4, base_ch*2, n_res=2)
        self.dec2 = DecoderBlock(base_ch*2,  base_ch*2, base_ch,   n_res=2)
        self.dec1 = DecoderBlock(base_ch,    base_ch,   base_ch,   n_res=1)

        # Выходной слой
        self.head = nn.Sequential(
            nn.Conv2d(base_ch, base_ch // 2, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_ch // 2, out_ch, 1),
            nn.Sigmoid(),   # выход в [0, 1]
        )

    def forward(self, x):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)

        x = self.bottleneck(x)

        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)

        return self.head(x)


# --- Создание модели с DataParallel ---
model_raw = ResAttUNet(in_ch=2, out_ch=2, base_ch=64).to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"Используем {torch.cuda.device_count()} GPU через DataParallel")
    model = nn.DataParallel(model_raw)
else:
    print("Одна GPU или CPU")
    model = model_raw

# Подсчёт параметров
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Параметров модели: {n_params:,}")

Используем 2 GPU через DataParallel
Параметров модели: 96,717,922


In [8]:
# ЯЧЕЙКА 7 — Функции потерь


# ---- SSIM Loss ----
class SSIMLoss(nn.Module):
    """Differentiable SSIM loss (1 - SSIM), работает поканально."""

    def __init__(self, window_size=11, sigma=1.5):
        super().__init__()
        self.window_size = window_size
        kernel = self._gaussian_kernel(window_size, sigma)
        # [1, 1, W, W] — будем применять поканально
        self.register_buffer("kernel", kernel)

    @staticmethod
    def _gaussian_kernel(size, sigma):
        coords = torch.arange(size, dtype=torch.float32) - size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g /= g.sum()
        kernel = g.outer(g)
        return kernel.unsqueeze(0).unsqueeze(0)

    def forward(self, pred, target):
        C1, C2 = 0.01**2, 0.03**2
        B, C, H, W = pred.shape
        kernel = self.kernel.expand(C, 1, -1, -1)
        pad    = self.window_size // 2

        mu1 = F.conv2d(pred,   kernel, padding=pad, groups=C)
        mu2 = F.conv2d(target, kernel, padding=pad, groups=C)

        mu1_sq = mu1 ** 2
        mu2_sq = mu2 ** 2
        mu12   = mu1 * mu2

        sig1 = F.conv2d(pred**2,        kernel, padding=pad, groups=C) - mu1_sq
        sig2 = F.conv2d(target**2,      kernel, padding=pad, groups=C) - mu2_sq
        sig12= F.conv2d(pred * target,  kernel, padding=pad, groups=C) - mu12

        ssim_map = ((2*mu12 + C1) * (2*sig12 + C2)) / \
                   ((mu1_sq + mu2_sq + C1) * (sig1 + sig2 + C2))

        return 1.0 - ssim_map.mean()


# ---- Perceptual Loss (VGG-based, адаптирован под 2 канала) ----
class PerceptualLoss(nn.Module):
    """
    Извлекает признаки из VGG16 (relu1_2, relu2_2).
    2-канальный вход расширяется до 3 каналов репликацией среднего.
    """

    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        # relu1_2 и relu2_2
        self.slice1 = nn.Sequential(*list(vgg.features)[:4]).eval()
        self.slice2 = nn.Sequential(*list(vgg.features)[4:9]).eval()
        for p in self.parameters():
            p.requires_grad_(False)

    def _to3ch(self, x):
        """[B, 2, H, W] → [B, 3, H, W]: среднее как 3-й канал."""
        mean_ch = x.mean(dim=1, keepdim=True)
        return torch.cat([x, mean_ch], dim=1)

    def forward(self, pred, target):
        p = self._to3ch(pred)
        t = self._to3ch(target)
        loss = 0.0
        for slc in [self.slice1, self.slice2]:
            p = slc(p)
            t = slc(t)
            loss += F.l1_loss(p, t)
        return loss


# ---- Комбинированная функция потерь ----
class CombinedLoss(nn.Module):
    def __init__(self, lambda_l1=1.0, lambda_ssim=0.3, lambda_perc=0.05):
        super().__init__()
        self.l1   = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.perc = PerceptualLoss().to(DEVICE)
        self.lam_l1   = lambda_l1
        self.lam_ssim = lambda_ssim
        self.lam_perc = lambda_perc

    def forward(self, pred, target):
        l1   = self.l1(pred, target)
        ssim = self.ssim(pred, target)
        perc = self.perc(pred, target)
        total = self.lam_l1 * l1 + self.lam_ssim * ssim + self.lam_perc * perc
        return total, {"l1": l1.item(), "ssim": ssim.item(), "perc": perc.item()}


criterion = CombinedLoss(
    lambda_l1=LAMBDA_L1,
    lambda_ssim=LAMBDA_SSIM,
    lambda_perc=LAMBDA_PERC,
).to(DEVICE)

print("Функция потерь создана.")

Функция потерь создана.


In [9]:
# ЯЧЕЙКА 8 — Оптимизатор и scheduler


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=SCHEDULER_PATIENCE,
    min_lr=1e-6,
    verbose=True,
)


In [10]:
# ЯЧЕЙКА 9 — Вспомогательная функция визуализации


def visualize_batch(noisy, clean, pred, epoch, save_dir=VIZ_DIR, n=N_VIZ_SAMPLES):
    """
    Выводит и сохраняет сравнение:
    [зашумлённое | чистое | предсказанное] для первых n примеров.
    Каналы: показываем канал R (index 0).
    """
    n = min(n, noisy.shape[0])
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    titles = ["Вход (с полосами)", "Эталон (чистое)", "Предсказание"]
    for i in range(n):
        imgs = [
            noisy[i, 0].cpu().numpy(),
            clean[i, 0].cpu().numpy(),
            pred[i, 0].detach().cpu().numpy(),
        ]
        for j, (img, title) in enumerate(zip(imgs, titles)):
            axes[i, j].imshow(img, cmap="gray", vmin=0, vmax=1)
            axes[i, j].set_title(f"{title} (эпоха {epoch})", fontsize=9)
            axes[i, j].axis("off")

    plt.tight_layout()
    plt.show()
    plt.close()

In [11]:
import sys

class Tee:
    def __init__(self, filename, mode="w"):
        self.file   = open(filename, mode)
        self.stdout = sys.stdout

    def write(self, data):
        self.file.write(data)
        self.stdout.write(data)

    def flush(self):
        self.file.flush()
        self.stdout.flush()

    def close(self):
        self.file.close()

tee = Tee("training_log(13).txt")

sys.stdout = tee
sys.stderr = tee

In [ ]:
# ЯЧЕЙКА 10 — Цикл обучения


best_val_loss = float("inf")
history = {"train": [], "val": []}

# Хранение примеров для визуализации (берём из val)
viz_noisy, viz_clean = next(iter(val_loader))
viz_noisy = viz_noisy.to(DEVICE)
viz_clean = viz_clean.to(DEVICE)

print(f"Начинаем обучение | {NUM_EPOCHS} эпох | batch={BATCH_SIZE}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()

    # -------- Train --------
    model.train()
    train_loss = 0.0
    for step, (noisy, clean) in enumerate(train_loader):
        noisy = noisy.to(DEVICE, non_blocking=True)
        clean = clean.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        pred = model(noisy)
        loss, parts = criterion(pred, clean)
        loss.backward()

        # Gradient clipping для стабильности
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------- Validation --------
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for noisy, clean in val_loader:
            noisy = noisy.to(DEVICE, non_blocking=True)
            clean = clean.to(DEVICE, non_blocking=True)
            pred  = model(noisy)
            loss, _ = criterion(pred, clean)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    history["train"].append(train_loss)
    history["val"].append(val_loss)

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]["lr"]
    print(f"Эпоха [{epoch:3d}/{NUM_EPOCHS}] | "
          f"Train: {train_loss:.5f} | Val: {val_loss:.5f} | "
          f"LR: {lr_now:.2e} | {elapsed:.1f}с")

    # Сохранение лучших весов
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        ckpt_path = os.path.join(CHECKPOINT_DIR, "best_model.pth")
        torch.save({
            "epoch":      epoch,
            "model_state": model_raw.state_dict(),   # сохраняем без DataParallel
            "optimizer":  optimizer.state_dict(),
            "val_loss":   val_loss,
        }, ckpt_path)
        print(f"  ✓ Лучшая модель сохранена (val={val_loss:.5f})")

    # Визуализация
    if epoch % VIZ_EVERY == 0 or epoch == 1:
        with torch.no_grad():
            viz_pred = model(viz_noisy)
        visualize_batch(viz_noisy, viz_clean, viz_pred, epoch)

print(f"\nОбучение завершено. Лучший val loss: {best_val_loss:.5f}")

In [23]:
def load_best_model(model_raw, checkpoint_path=None):
    if checkpoint_path is None:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model13.pth")
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model_raw.load_state_dict(ckpt["model_state"])
    print(f"Загружены веса из '{checkpoint_path}' (эпоха {ckpt['epoch']}, "
          f"val={ckpt['val_loss']:.5f})")
    return model_raw


model_raw = load_best_model(model_raw)
model_raw.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints13/best_model13.pth'

In [25]:
def infer_image(model, img_np_2ch: np.ndarray, tile_size=256, overlap=32) -> np.ndarray:
    """
    Применяет модель к двухканальному изображению произвольного размера
    с помощью перекрывающихся тайлов (tile inference).

    Параметры:
        model        — обученная модель (без DataParallel, .eval())
        img_np_2ch   — numpy массив [2, H, W] или [H, W, 2], float32/uint16/uint8
        tile_size    — размер тайла (должен совпадать с IMG_SIZE обучения)
        overlap      — перекрытие в пикселях (уменьшает граничные артефакты)

    Возвращает:
        numpy массив [2, H, W], float32, диапазон [0, 1]
    """
    model.eval()

    # --- Нормализация входа ---
    img = np.array(img_np_2ch, dtype=np.float32)
    if img.ndim == 3 and img.shape[2] == 2:
        img = img.transpose(2, 0, 1)   # [H, W, 2] → [2, H, W]

    assert img.ndim == 3 and img.shape[0] == 2, \
        f"Ожидается [2, H, W] или [H, W, 2], получено {img.shape}"

    # Нормализуем в [0, 1] если нужно
    if img.max() > 1.0:
        img = img / img.max()

    C, H, W = img.shape
    step    = tile_size - overlap * 2

    result  = np.zeros_like(img)
    weights = np.zeros((H, W), dtype=np.float32)

    # Мягкое взвешивание (Hann window) для сглаживания стыков
    hann_1d = np.hanning(tile_size).astype(np.float32)
    hann_2d = np.outer(hann_1d, hann_1d)

    with torch.no_grad():
        y = 0
        while y < H:
            x = 0
            while x < W:
                y1 = min(y, H - tile_size)
                x1 = min(x, W - tile_size)
                y2, x2 = y1 + tile_size, x1 + tile_size

                tile = img[:, y1:y2, x1:x2]
                t    = torch.from_numpy(tile).unsqueeze(0).to(DEVICE)
                out  = model(t).squeeze(0).cpu().numpy()

                result[:, y1:y2, x1:x2] += out * hann_2d[np.newaxis]
                weights[y1:y2, x1:x2]   += hann_2d

                x += step
                if x1 + tile_size >= W:
                    break
            y += step
            if y1 + tile_size >= H:
                break

    # Нормируем по весам
    weights = np.maximum(weights, 1e-6)
    result /= weights[np.newaxis]
    result  = np.clip(result, 0.0, 1.0)

    return result


# --- Загрузка реального снимка ---
print(f"Загрузка {LF_PATH}...")
lf_raw = np.load(LF_PATH)
print(f"  Форма: {lf_raw.shape}, dtype: {lf_raw.dtype}, "
      f"min: {lf_raw.min():.3f}, max: {lf_raw.max():.3f}")

# Применяем модель
restored = infer_image(model_raw, lf_raw, tile_size=IMG_SIZE, overlap=32)
print(f"Результат: форма {restored.shape}, min={restored.min():.4f}, max={restored.max():.4f}")

print(f"Восстановленный снимок сохранён: {output_path}")

AssertionError: Ожидается [2, H, W] или [H, W, 2], получено (131, 131)

In [26]:
def infer_k(model, img_P: np.ndarray, tile_size=512, overlap=32) -> np.ndarray:
    """
    Применяет модель к одноканальному изображению P произвольного размера.
    Возвращает предсказанную маску K того же размера [H, W] в диапазоне [0,1].

    Параметры:
        model     — модель KPredictor (без DataParallel, .eval())
        img_P     — numpy массив [H, W] или [1, H, W], float32/float64/uint*
        tile_size — размер тайла (лучше тот же, что при обучении, напр. 512)
        overlap   — перекрытие тайлов (уменьшает артефакты на стыках)

    Возвращает:
        numpy массив [H, W], float32, значения [0, 1]
    """
    model.eval()

    # --- Приведение к формату [1, H, W] и нормализация ---
    img = np.array(img_P, dtype=np.float32)
    if img.ndim == 2:
        img = img[np.newaxis, :, :]          # [H, W] → [1, H, W]
    elif img.ndim == 3 and img.shape[0] != 1:
        # возможно, [H, W, 1] → транспонируем
        if img.shape[-1] == 1:
            img = img.transpose(2, 0, 1)     # → [1, H, W]

    assert img.ndim == 3 and img.shape[0] == 1, \
        f"Ожидается одноканальное изображение, получена форма {img.shape}"

    # Нормализация в [0, 1]
    if img.max() > 1.0:
        img = img / img.max()

    C, H, W = img.shape
    step = tile_size - overlap * 2

    result = np.zeros((H, W), dtype=np.float32)
    weights = np.zeros((H, W), dtype=np.float32)

    # Окно Ханна для сглаживания стыков
    hann_1d = np.hanning(tile_size).astype(np.float32)
    hann_2d = np.outer(hann_1d, hann_1d)

    with torch.no_grad():
        y = 0
        while y < H:
            x = 0
            while x < W:
                y1 = min(y, H - tile_size)
                x1 = min(x, W - tile_size)
                y2, x2 = y1 + tile_size, x1 + tile_size

                tile = img[:, y1:y2, x1:x2]                     # [1, tile, tile]
                t = torch.from_numpy(tile).unsqueeze(0).to(DEVICE)  # [1, 1, tile, tile]
                out = model(t).squeeze(0).squeeze(0).cpu().numpy()  # [tile, tile]

                result[y1:y2, x1:x2] += out * hann_2d
                weights[y1:y2, x1:x2] += hann_2d

                x += step
                if x1 + tile_size >= W:
                    break
            y += step
            if y1 + tile_size >= H:
                break

    weights = np.maximum(weights, 1e-6)
    result /= weights
    result = np.clip(result, 0.0, 1.0)

    return result

In [28]:
lf_P = np.load(LF_PATH)                     # [131, 131]
K_pred = infer_k(model_raw, lf_P, tile_size=512, overlap=32)

print(f"Маска K: форма {K_pred.shape}, min={K_pred.min():.4f}, max={K_pred.max():.4f}")

RuntimeError: Given groups=1, weight of size [64, 2, 3, 3], expected input[1, 1, 131, 131] to have 2 channels, but got 1 channels instead